# Stage 1 — LIME Feature Extraction

**Reads:** Input texts defined in this notebook  
**Writes:** `outputs/lime_results.json`

Each record saved:
```json
{
  "text": "...",
  "fused_text": "...",
  "lime_features": [["hypertension", 0.42], ...]
}
```

> **Tip:** Run cells top to bottom on first run.  
> If `outputs/lime_results.json` already exists the checkpoint cell will warn you — set `FORCE_RERUN = True` to overwrite it.

## 1. Imports

In [1]:
from lime.lime_text import LimeTextExplainer

from config import (
    CLASS_NAMES,
    LIME_NUM_FEATURES,
    LIME_NUM_SAMPLES,
    LIME_RESULTS_PATH,
)
from model_loaders import load_classifier, load_ner_pipeline
from pipeline_helpers import (
    checkpoint_exists,
    load_checkpoint,
    make_lime_predictor,
    merge_entities,
    save_checkpoint,
)

c:\Users\vimal\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

In [2]:
# Set True to re-run even if lime_results.json already exists
FORCE_RERUN = False

# ── Add / edit your input texts here ────────────────────────────────────────
TEXTS = [
    (
        "Intravenous nicardipine: an effective new agent for the treatment of severe "
        "hypertension. Fifty-six patients with severe hypertension were treated with "
        "intravenous nicardipine for infusion periods of eight to twenty-four hours. "
        "Each patient achieved satisfactory blood pressure control during the infusion "
        "period with a mean controlling dose of 7.85 mg/hr."
    ),
    # Add more texts below:
    # "...",
]

## 3. Checkpoint check

In [3]:
if checkpoint_exists(LIME_RESULTS_PATH) and not FORCE_RERUN:
    print(f"⚠️  Checkpoint found at '{LIME_RESULTS_PATH}'.")
    print("    Set FORCE_RERUN = True in the cell above to overwrite.")
    print("    Loading existing results …")
    results = load_checkpoint(LIME_RESULTS_PATH)
else:
    results = None
    print("No checkpoint found (or FORCE_RERUN=True). Will run LIME.")

No checkpoint found (or FORCE_RERUN=True). Will run LIME.


## 4. Load models

In [4]:
if results is None:
    classifier_model, classifier_pipeline = load_classifier()
    ner_pipeline = load_ner_pipeline()

[Loader] Loading classifier from 'C:/Users/vimal/OneDrive/Documents/Uni/BTP/User-Adaptive-XAI/Models/my_medical_model' …


Device set to use cpu


[Loader] Classifier ready.

[Loader] Loading NER model 'd4data/biomedical-ner-all' …


Device set to use cpu


[Loader] NER model ready.



## 5. Run LIME

In [5]:
if results is None:
    explainer = LimeTextExplainer(class_names=CLASS_NAMES)
    predictor = make_lime_predictor(classifier_model, classifier_pipeline)

    results = []
    for i, text in enumerate(TEXTS):
        print(f"\nProcessing text {i + 1}/{len(TEXTS)} …")

        # Fuse multi-word biomedical entities before LIME
        fused_text = merge_entities(text, ner_pipeline)
        print(f"  Fused text preview: {fused_text[:120]}…")

        exp = explainer.explain_instance(
            fused_text,
            predictor,
            num_features=LIME_NUM_FEATURES,
            num_samples=LIME_NUM_SAMPLES,
        )

        # Restore underscores → spaces for downstream readability
        lime_features = [
            [w.replace("_", " "), float(score)]
            for w, score in exp.as_list()
        ]

        print(f"  Top features: {[f[0] for f in lime_features]}")

        results.append({
            "text":          text,
            "fused_text":    fused_text,
            "lime_features": lime_features,
        })

    print("\n✅ LIME complete.")


Processing text 1/1 …
  Fused text preview: Intravenous nicardipine: an effective new agent for the treatment of severe hypertension. Fifty-six patients with severe…
  Top features: ['hypertension', 'nicardipine', 'blood pressure control', 'eight', 'dose', 'intravenous']

✅ LIME complete.


## 6. Inspect results

In [6]:
for i, r in enumerate(results):
    print(f"\n── Text {i + 1} ──────────────────────────")
    print(f"Text preview : {r['text'][:100]}…")
    print(f"Top features : {r['lime_features']}")


── Text 1 ──────────────────────────
Text preview : Intravenous nicardipine: an effective new agent for the treatment of severe hypertension. Fifty-six …
Top features : [['hypertension', -0.010406451079492784], ['nicardipine', -0.0098294080789622], ['blood pressure control', -0.007683463026796646], ['eight', -0.0032467115224290516], ['dose', -0.002714869612920816], ['intravenous', -0.002681827723056484]]


## 7. Save checkpoint

In [7]:
save_checkpoint(results, LIME_RESULTS_PATH)
print(f"\n➡️  Continue to notebook 02_ontology.ipynb")

[Checkpoint] Saved 1 records → 'outputs\lime_results.json'

➡️  Continue to notebook 02_ontology.ipynb
